In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [9]:
import numpy as np
import pandas as pd
import os
import warnings
import gc
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import xgboost as xgb
warnings.filterwarnings("ignore")
print("Imports OK ✓")

Imports OK ✓


In [10]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "kende23"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "be7a74e194c6b7c0d666a4938cfee49142c7d603"
mlflow.set_tracking_uri("https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow")
dagshub.init(repo_owner="kende23", repo_name="ieee-cis-fraud-detection", mlflow=True)

experiment_name = "XGBoost_Magic"
if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(experiment_name)
mlflow.set_experiment(experiment_name)
print("MLflow connected ✓")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=50329404-1a67-4924-aa7d-b3f9e59fe7d4&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=5a8d4ab289c09e1b613c804fcf419dc358a5ea3ca54c31f99f2655bc3e49cb0b




Accessing as kende23

Initialized MLflow to track repo "kende23/ieee-cis-fraud-detection"

Repository kende23/ieee-cis-fraud-detection initialized!

MLflow connected ✓


In [12]:
path = "/kaggle/input/competitions/ieee-fraud-detection/"

print("Loading...")
train_tr = pd.read_csv(path + "train_transaction.csv")
train_id = pd.read_csv(path + "train_identity.csv")
test_tr  = pd.read_csv(path + "test_transaction.csv")
test_id  = pd.read_csv(path + "test_identity.csv")

train_id.columns = [c.replace("-", "_") for c in train_id.columns]
test_id.columns  = [c.replace("-", "_") for c in test_id.columns]

train = train_tr.merge(train_id, on="TransactionID", how="left")
test  = test_tr.merge(test_id,   on="TransactionID", how="left")
print(f"train: {train.shape}, test: {test.shape}")

v_cols = [1,3,4,6,8,11,
          13,14,17,20,23,26,27,30,
          36,37,40,41,44,47,48,
          54,56,59,62,65,67,68,70,
          76,78,80,82,86,88,89,91,
          107,108,111,115,117,120,121,123,
          124,127,129,130,136,
          138,139,142,147,156,162,165,
          166,167,168,176,178,180,182,
          187,203,207,209,
          210,211,212,216,217,220,
          221,222,223,224,225,226,
          228,229,235,
          235,240,
          258,272,274,277,
          294,
          303,305,307,309,310,320]
v_cols = [f"V{v}" for v in v_cols if f"V{v}" in train.columns]
print(f"Selected {len(v_cols)} V columns")

base_cols = ["TransactionID","isFraud","TransactionDT","TransactionAmt",
             "ProductCD","card1","card2","card3","card4","card5","card6",
             "addr1","addr2","dist1","dist2","P_emaildomain","R_emaildomain",
             "C1","C2","C3","C4","C5","C6","C7","C8","C9","C10","C11","C12","C13","C14",
             "D1","D2","D3","D4","D5","D6","D7","D8","D9","D10","D11","D12","D13","D14","D15",
             "M1","M2","M3","M4","M5","M6","M7","M8","M9"]

train_cols = [c for c in base_cols if c in train.columns] + v_cols
test_cols  = [c for c in base_cols if c in test.columns and c != "isFraud"] + v_cols

train = train[[c for c in train_cols if c in train.columns]]
test  = test[[c for c in test_cols  if c in test.columns]]

print(f"After column selection: train={train.shape}, test={test.shape}")
print(f"Fraud rate: {train['isFraud'].mean():.4f}")

Loading...
train: (590540, 434), test: (506691, 433)
Selected 96 V columns
After column selection: train=(590540, 151), test=(506691, 150)
Fraud rate: 0.0350


In [13]:
print("=== FEATURE ENGINEERING (cdeotte approach) ===")

# ── combine train and test for aggregations ──
# this is KEY — avoids leakage on test set
# test rows get NaN for isFraud which is fine
print("Combining train+test for aggregations...")
train["is_train"] = 1
test["is_train"]  = 0
combined = pd.concat([train, test], ignore_index=True, sort=False)
print(f"Combined: {combined.shape}")

# ── 1. time features ──
print("Adding time features...")
combined["TransactionDay"]  = (combined["TransactionDT"] / 86400).astype(np.float32)
combined["TransactionHour"] = ((combined["TransactionDT"] / 3600) % 24).astype(np.float32)

# ── 2. D column normalization ──
print("Normalizing D columns...")
combined["D1n"]  = combined["TransactionDay"] - combined["D1"]
combined["D2n"]  = combined["TransactionDay"] - combined["D2"]
combined["D3n"]  = combined["TransactionDay"] - combined["D3"]
combined["D4n"]  = combined["TransactionDay"] - combined["D4"]
combined["D10n"] = combined["TransactionDay"] - combined["D10"]
combined["D15n"] = combined["TransactionDay"] - combined["D15"]

# ── 3. cents feature ──
combined["cents"] = (combined["TransactionAmt"] - np.floor(combined["TransactionAmt"])).astype(np.float32)

# ── 4. email provider ──
combined["P_email_provider"] = combined["P_emaildomain"].str.split(".").str[0].fillna("unknown")
combined["R_email_provider"] = combined["R_emaildomain"].str.split(".").str[0].fillna("unknown")

# ── 5. frequency encoding (cdeotte's FE features) ──
print("Adding frequency encoding...")
for col in ["addr1", "card1", "card2", "card3", "P_emaildomain"]:
    freq = combined[col].value_counts(dropna=False)
    combined[f"{col}_FE"] = combined[col].map(freq).astype(np.float32)
    print(f"  {col}_FE: {combined[col].nunique()} unique values")

# ── 6. UID = card1 + addr1 + P_emaildomain (cdeotte's magic) ──
print("Creating UID...")
combined["uid"] = (
    combined["card1"].astype(str) + "_" +
    combined["addr1"].astype(str) + "_" +
    combined["P_emaildomain"].astype(str)
)
combined["uid2"] = (
    combined["card1"].astype(str) + "_" +
    combined["addr1"].astype(str)
)
print(f"  Unique UIDs:  {combined['uid'].nunique()}")
print(f"  Unique UID2s: {combined['uid2'].nunique()}")

# ── 7. frequency encoding for UIDs ──
uid_freq  = combined["uid"].value_counts(dropna=False)
uid2_freq = combined["uid2"].value_counts(dropna=False)
combined["uid_FE"]  = combined["uid"].map(uid_freq).astype(np.float32)
combined["uid2_FE"] = combined["uid2"].map(uid2_freq).astype(np.float32)

# ── 8. aggregation features (exact cdeotte features) ──
print("Adding aggregation features...")

# TransactionAmt aggregations
for grp in ["card1", "uid2"]:
    amt_mean = combined.groupby(grp)["TransactionAmt"].mean()
    amt_std  = combined.groupby(grp)["TransactionAmt"].std()
    combined[f"TransactionAmt_{grp}_mean"] = combined[grp].map(amt_mean).astype(np.float32)
    combined[f"TransactionAmt_{grp}_std"]  = combined[grp].map(amt_std).astype(np.float32)
    print(f"  TransactionAmt_{grp} agg done")

# D9 aggregations
for grp in ["card1", "uid2"]:
    d9_mean = combined.groupby(grp)["D9"].mean()
    d9_std  = combined.groupby(grp)["D9"].std()
    combined[f"D9_{grp}_mean"] = combined[grp].map(d9_mean).astype(np.float32)
    combined[f"D9_{grp}_std"]  = combined[grp].map(d9_std).astype(np.float32)
    print(f"  D9_{grp} agg done")

# D11 aggregations  
for grp in ["card1", "uid2"]:
    d11_mean = combined.groupby(grp)["D11"].mean()
    d11_std  = combined.groupby(grp)["D11"].std()
    combined[f"D11_{grp}_mean"] = combined[grp].map(d11_mean).astype(np.float32)
    combined[f"D11_{grp}_std"]  = combined[grp].map(d11_std).astype(np.float32)
    print(f"  D11_{grp} agg done")

# ── 9. drop uid columns — don't use directly ──
combined = combined.drop(columns=["uid", "uid2"])

# ── split back into train and test ──
print("\nSplitting back...")
train = combined[combined["is_train"] == 1].drop(columns=["is_train"])
test  = combined[combined["is_train"] == 0].drop(columns=["is_train", "isFraud"])
print(f"train: {train.shape}, test: {test.shape}")

del combined
gc.collect()

X = train.drop(columns=["isFraud", "TransactionID"])
y = train["isFraud"]
test = test.drop(columns=["TransactionID"])

# align columns
common_cols = [c for c in X.columns if c in test.columns]
X    = X[common_cols]
test = test[common_cols]
assert list(X.columns) == list(test.columns)
cols_after_engineering = X.shape[1]
print(f"\nFinal: X={X.shape}, test={test.shape}")
print("Feature engineering done ✓")

=== FEATURE ENGINEERING (cdeotte approach) ===
Combining train+test for aggregations...


InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [ ]:
print("=== FEATURE SELECTION ===")
print(f"Starting: X={X.shape}")

# drop high missing >90%
missing_rate = X.isnull().mean()
high_missing = missing_rate[missing_rate > 0.9].index.tolist()
print(f"Dropping {len(high_missing)} high missing cols")
X    = X.drop(columns=high_missing)
test = test.drop(columns=high_missing)

# drop near-zero variance
num_cols_temp = X.select_dtypes(include=[np.number]).columns
low_var = [c for c in num_cols_temp if X[c].std() < 0.01]
print(f"Dropping {len(low_var)} low variance cols: {low_var}")
X    = X.drop(columns=low_var)
test = test.drop(columns=low_var)

# drop high cardinality categoricals only
cat_cols_temp = X.select_dtypes(include=["object"]).columns
high_card = [c for c in cat_cols_temp if X[c].nunique() > 200]
print(f"Dropping {len(high_card)} high cardinality cols: {high_card}")
X    = X.drop(columns=high_card)
test = test.drop(columns=high_card)

# NOTE: no aggressive correlation dropping this time
# cdeotte kept ~250 features, we trust XGBoost to handle correlations

assert list(X.columns) == list(test.columns)
cols_after_cleaning = X.shape[1]
final_feature_count = X.shape[1]
final_cols = X.columns.tolist()
print(f"\nFinal: X={X.shape}")
print("✓ No aggressive correlation dropping — XGBoost handles this internally")

In [ ]:
print("=== MEMORY REDUCTION ===")
def reduce_mem(df, name):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    print(f"  {name}: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
    return df

X    = reduce_mem(X,    "X")
test = reduce_mem(test, "test")
print(X.dtypes.value_counts())

In [ ]:
print("=== TIME-BASED SPLIT ===")
X = X.sort_values("TransactionDT")
y = y[X.index].reset_index(drop=True)
X = X.reset_index(drop=True)

split_point = X["TransactionDT"].quantile(0.8)
print(f"Split at 80th percentile: {split_point:.0f}")

train_mask = X["TransactionDT"] <= split_point
val_mask   = X["TransactionDT"] >  split_point

X_train = X[train_mask].reset_index(drop=True)
X_val   = X[val_mask].reset_index(drop=True)
y_train = y[train_mask].reset_index(drop=True)
y_val   = y[val_mask].reset_index(drop=True)

print(f"X_train: {X_train.shape}, fraud: {y_train.mean():.4f}")
print(f"X_val:   {X_val.shape},   fraud: {y_val.mean():.4f}")

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")
print(f"Cat cols: {cat_cols}")

del X
gc.collect()
print("Freed X ✓")